# Dirac delta in orthogonal coordinates

A short computational check of Problems 1.2–1.3.  The numerics do not replace the distributional proof; they test its two claims:

1. the Gaussian sequence samples a smooth test function at the source point;
2. each proposed charge density has the required total charge.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss

plt.style.use('seaborn-v0_8-whitegrid')

# Parameters
q0 = np.array([0.30, -0.40, 0.20])       # (u', v', w')
metric_factors = np.array([1.40, 0.80, 1.70])  # (U', V', W')
alphas = 2.0 ** -np.arange(1, 8)

Q = 7.0
lambda_line = 2.5
R = 2.3
b = 1.4

## 1.2 — Gaussian sequence

Locally,

$$ds^2=\frac{du^2}{U'^2}+\frac{dv^2}{V'^2}+\frac{dw^2}{W'^2},\qquad d^3x=\frac{du\,dv\,dw}{UVW}.$$

Thus $D_\alpha\,d^3x$ is locally the product of three normalized Gaussians, with coordinate widths $\alpha U'$, $\alpha V'$, and $\alpha W'$.  For any smooth $f$,

$$\int D_\alpha f\,d^3x \longrightarrow f(u',v',w').$$

Gauss–Hermite quadrature evaluates that integral directly.

In [ ]:
def test_function(u, v, w):
    envelope = np.exp(-0.20 * (u**2 + 2.0*v**2 + 0.50*w**2))
    return envelope * (1.0 + 0.10*u - 0.05*v + 0.03*u*w)

def gaussian_action(alpha, order=24):
    # E[f(q0 + alpha * metric_factors * Z)],  Z ~ N(0, I)
    nodes, weights = hermgauss(order)
    u = q0[0] + np.sqrt(2.0) * alpha * metric_factors[0] * nodes[:, None, None]
    v = q0[1] + np.sqrt(2.0) * alpha * metric_factors[1] * nodes[None, :, None]
    w = q0[2] + np.sqrt(2.0) * alpha * metric_factors[2] * nodes[None, None, :]
    W3 = weights[:, None, None] * weights[None, :, None] * weights[None, None, :]
    return np.sum(W3 * test_function(u, v, w)) / np.pi**1.5

target = test_function(*q0)
actions = np.array([gaussian_action(alpha) for alpha in alphas])
errors = np.abs(actions - target)

pd.DataFrame({
    'alpha': alphas,
    'Gaussian action': actions,
    'f(source)': target,
    'absolute error': errors,
})

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.loglog(alphas, errors, 'o-', lw=2, color='#2455A4')
ax.invert_xaxis()
ax.set(xlabel=r'$\alpha$', ylabel='absolute error',
       title=r'$\int D_\alpha f\,d^3x \to f(\mathbf{x}^{\prime})$')
plt.show()

The error is $O(\alpha^2)$: the normalized Gaussian is even, so its linear moment vanishes.

## 1.3 — charge normalization

Let $g_\epsilon(t)$ be a normalized Gaussian approximation to $\delta(t)$.  The proposed densities are

$$
\rho_a=\frac{Q}{4\pi R^2}\delta(r-R),\qquad
\rho_b=\frac{\lambda}{2\pi b}\delta(s-b),
$$

$$
\rho_c=\frac{Q}{\pi R^2}\,\delta(z)\Theta(R-s),\qquad
\rho_d=\frac{Q}{\pi R^2}\,\frac{\delta(\theta-\pi/2)}{r}\Theta(R-r).
$$

The next cell replaces each delta by $g_\epsilon$ and computes recovered charge / specified charge.  A standard-normal variable keeps every integral well resolved as $\epsilon\to0$.

In [ ]:
def phi(t):
    return np.exp(-0.5*t**2) / np.sqrt(2.0*np.pi)

t = np.linspace(-8.0, 8.0, 40_001)
epsilons = 2.0 ** -np.arange(0, 7)
rows = []

for eps in epsilons:
    # Physical radial coordinates require r,s >= 0.
    shell_mask = (R + eps*t) >= 0.0
    cyl_mask = (b + eps*t) >= 0.0
    theta_mask = np.abs(eps*t) <= np.pi/2

    shell_ratio = np.trapezoid(
        ((R + eps*t[shell_mask]) / R)**2 * phi(t[shell_mask]),
        t[shell_mask],
    )
    cylinder_ratio = np.trapezoid(
        ((b + eps*t[cyl_mask]) / b) * phi(t[cyl_mask]),
        t[cyl_mask],
    )
    flat_disc_ratio = np.trapezoid(phi(t), t)
    spherical_disc_ratio = np.trapezoid(
        np.cos(eps*t[theta_mask]) * phi(t[theta_mask]),
        t[theta_mask],
    )

    rows.append([eps, shell_ratio, cylinder_ratio,
                 flat_disc_ratio, spherical_disc_ratio])

charge_check = pd.DataFrame(
    rows,
    columns=['epsilon', 'sphere shell', 'cylinder surface',
             'disc: cylindrical', 'disc: spherical'],
)
charge_check

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.2))
for column in charge_check.columns[1:]:
    ax.semilogx(charge_check['epsilon'], charge_check[column], 'o-', label=column)

ax.axhline(1.0, color='black', lw=1, ls='--')
ax.invert_xaxis()
ax.set(xlabel=r'$\epsilon$', ylabel='recovered / specified charge',
       title='Normalization of all four densities')
ax.legend(frameon=True)
plt.show()

All four ratios tend to $1$.  The small finite-width bias in the shell comes from averaging the curved measure $r^2$ across the Gaussian; it vanishes in the delta limit.

In [ ]:
# Short machine-checkable conclusion
assert errors[-1] < errors[0]
assert np.max(np.abs(charge_check.iloc[-1, 1:].to_numpy() - 1.0)) < 2.0e-4
print('Passed: sampling converges and every charge ratio tends to 1.')